# Bases de données — Exercice 7 : location de voitures (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** clé primaire, clé étrangère et suppression, `UPDATE`, `SELECT ... WHERE`, `COUNT`, `IS NULL`, jointures sur plusieurs relations, comparaison de dates.


Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

> Afin de gérer les réservations de ses véhicules, une société de location de voiture dispose d'une base de données dont le schéma relationnel est le suivant.
>
> - Client (<u>idC</u>, nom, prénom, email)
> - Location (<u>idL</u>, idCli\*, idVehi\*, dateDeb, dateFin)
> - Véhicule (<u>idV</u>, modele\*, immatr, nbKm)
> - Modèle (<u>idM</u>, marque, année, moteur)
>
> (`Location.idCli` fait référence à `Client.idC`, `Location.idVehi` à `Véhicule.idV`, `Véhicule.modele` à `Modèle.idM`.)
>
> On suppose que ces relations ont été définies dans un SGBD basé sur le langage SQL. On admet de plus que les attributs dateDeb et dateFin de la relation Location sont au format DATETIME. Une date et heure au format DATETIME s'écrit à l'aide de la chaîne de caractères 'AAAA-MM-JJ HH:MM:SS'. Il est possible de comparer deux valeurs au format DATETIME à l'aide des opérateurs usuels (<, >=, etc.).

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

**Données inventées :** le sujet ne fournit pas de données. Pour pouvoir **tester** les requêtes, on crée ci-dessous la base avec quelques enregistrements imaginaires (les clients, adresses électroniques et immatriculations sont fictifs).

⚠️ **H2 et `DATETIME` :** H2 (version 2) ne connaît pas le type `DATETIME` (qui vient de MySQL) ; son équivalent est **`TIMESTAMP`**, utilisé ci-dessous. Les dates s'écrivent de la même façon, `'2025-11-01 08:00:00'`.

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")


base.execute("PRAGMA foreign_keys = ON")   # SQLite : activer la vérification des clés étrangères (H2 le fait toujours)

base.executescript("""
CREATE TABLE Client (idC INT, nom VARCHAR, prénom VARCHAR, email VARCHAR,
    PRIMARY KEY (idC));
CREATE TABLE Modèle (idM INT, marque VARCHAR, année INT, moteur VARCHAR,
    PRIMARY KEY (idM));
CREATE TABLE Véhicule (idV INT, modele INT, immatr VARCHAR, nbKm INT,
    PRIMARY KEY (idV),
    FOREIGN KEY (modele) REFERENCES Modèle(idM));
CREATE TABLE Location (idL INT, idCli INT, idVehi INT, dateDeb TIMESTAMP, dateFin TIMESTAMP,
    PRIMARY KEY (idL),
    FOREIGN KEY (idCli) REFERENCES Client(idC),
    FOREIGN KEY (idVehi) REFERENCES Véhicule(idV));

INSERT INTO Client VALUES (1345, 'Dupont', 'Marie', 'marie.dupont@exemple.fr');
INSERT INTO Client VALUES (1346, 'Durand', 'Paul', 'paul.durand@exemple.fr');
INSERT INTO Client VALUES (1347, 'Nguyen', 'Linh', 'linh.nguyen@exemple.fr');
INSERT INTO Client VALUES (1348, 'Moreau', 'Hugo', 'hugo.moreau@exemple.fr');

INSERT INTO Modèle VALUES (1, 'Renault', 2022, 'électrique');
INSERT INTO Modèle VALUES (2, 'Peugeot', 2020, 'essence');
INSERT INTO Modèle VALUES (3, 'Tesla', 2023, 'électrique');
INSERT INTO Modèle VALUES (4, 'Citroën', 2019, 'diesel');

INSERT INTO Véhicule VALUES (245, 2, 'AA-123-AA', 62000);
INSERT INTO Véhicule VALUES (246, 1, 'BB-456-BB', 18000);
INSERT INTO Véhicule VALUES (247, 3, 'CC-789-CC', 35000);
INSERT INTO Véhicule VALUES (248, 4, 'DD-012-DD', 125000);
INSERT INTO Véhicule VALUES (249, 1, 'EE-345-EE', 51000);
INSERT INTO Véhicule VALUES (250, 2, 'FF-678-FF', 9000);

INSERT INTO Location VALUES (9101, 1346, 246, '2025-09-03 09:00:00', '2025-09-05 18:00:00');
INSERT INTO Location VALUES (9102, 1347, 247, '2025-09-20 10:00:00', '2025-09-21 10:00:00');
INSERT INTO Location VALUES (9103, 1346, 249, '2025-09-28 08:00:00', '2025-10-02 12:00:00');
INSERT INTO Location VALUES (9104, 1348, 245, '2025-09-10 14:00:00', '2025-09-12 14:00:00');
INSERT INTO Location VALUES (9105, 1348, 246, '2025-10-15 09:00:00', '2025-10-16 09:00:00');
INSERT INTO Location VALUES (9201, 1347, 248, '2025-11-02 07:30:00', null);
INSERT INTO Location VALUES (9202, 1346, 247, '2025-11-04 11:00:00', null);
""")
affiche("SELECT * FROM Location")

idL | idCli | idVehi | dateDeb | dateFin | 
9101 | 1346 | 246 | 2025-09-03 09:00:00 | 2025-09-05 18:00:00 | 
9102 | 1347 | 247 | 2025-09-20 10:00:00 | 2025-09-21 10:00:00 | 
9103 | 1346 | 249 | 2025-09-28 08:00:00 | 2025-10-02 12:00:00 | 
9104 | 1348 | 245 | 2025-09-10 14:00:00 | 2025-09-12 14:00:00 | 
9105 | 1348 | 246 | 2025-10-15 09:00:00 | 2025-10-16 09:00:00 | 
9201 | 1347 | 248 | 2025-11-02 07:30:00 | None | 
9202 | 1346 | 247 | 2025-11-04 11:00:00 | None | 
(7 ligne(s))


---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Dans la relation Client, indiquer, en justifiant, un autre choix possible pour la clé primaire.

#### ✅ RÉSOLUTION

L'attribut **`email`** peut servir de clé primaire : une adresse électronique appartient à **une seule personne**, donc deux clients différents ne peuvent pas avoir la même adresse. Elle identifie chaque client de manière unique.

En revanche, `nom`, `prénom` ou le couple (`nom`, `prénom`) ne conviennent pas, à cause des homonymes.

#### 💡 EXPLICATION PÉDAGOGIQUE

- On demande une **autre clé candidate** que `idC` : un attribut (ou un groupe d'attributs) qui prend une valeur **différente pour chaque client**.
- Même raisonnement que dans l'exercice 4 : on écarte tout ce qui peut être partagé par deux personnes (nom, prénom).
- **Nuance utile :** l'adresse électronique est une clé possible, mais un client peut **changer** d'adresse. C'est pour cela qu'on préfère en général un identifiant artificiel comme `idC`, qui ne change jamais (sinon il faudrait aussi modifier toutes les locations qui y font référence).

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. À quelle condition est-il possible de supprimer un enregistrement dans la relation Véhicule ?

#### ✅ RÉSOLUTION

On ne peut supprimer un véhicule que s'il n'est **référencé par aucune location**, c'est-à-dire si son identifiant `idV` n'apparaît dans **aucun** enregistrement de la relation `Location` (attribut `idVehi`).

Sinon, la suppression violerait la **contrainte d'intégrité référentielle** : des locations feraient référence à un véhicule qui n'existe plus. Il faudrait d'abord supprimer (ou modifier) ces locations.

*Illustration (non demandée) :*

In [2]:
# Le véhicule 245 a déjà été loué : suppression refusée
try:
    affiche("DELETE FROM Véhicule WHERE idV = 245")
except sqlite3.IntegrityError as erreur:
    print("Refusé par le SGBD :", erreur)

# Le véhicule 250 n'a jamais été loué : suppression acceptée
affiche("DELETE FROM Véhicule WHERE idV = 250")
assert resultat("SELECT COUNT(*) FROM Véhicule WHERE idV = 250") == [(0,)]

Refusé par le SGBD : FOREIGN KEY constraint failed
Instruction exécutée.


#### 💡 EXPLICATION PÉDAGOGIQUE

- `Location.idVehi` est une **clé étrangère** qui fait référence à `Véhicule.idV`. La contrainte d'intégrité référentielle s'applique **dans les deux sens** :
  - à l'**ajout** d'une location : le véhicule doit exister ;
  - à la **suppression** d'un véhicule : aucune location ne doit encore y faire référence.
- La relation `Véhicule` n'est référencée que par `Location` : c'est la seule relation à vérifier. (La clé étrangère `modele` de `Véhicule` vers `Modèle` ne gêne pas la suppression d'un véhicule : elle empêcherait au contraire de supprimer un **modèle** utilisé par un véhicule.)
- Le SGBD **refuse** une suppression qui casserait la cohérence de la base : c'est un des services rendus par un SGBD.

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. Mme Dupont a loué une voiture le 1er novembre 2025 à 8h. L'instruction SQL suivante a permis l'enregistrement de cette location dans la base de données.
> ```sql
> INSERT INTO Location VALUES (9268, 1345, 245, '2025-11-01 08:00:00', null);
> ```
> Mme Dupont a restitué le véhicule le 3 novembre 2025 à 16h : quelle requête SQL permet de mettre à jour en conséquence la base de données ?

In [3]:
affiche("INSERT INTO Location VALUES (9268, 1345, 245, '2025-11-01 08:00:00', null)")
affiche("SELECT * FROM Location WHERE idL = 9268")

Instruction exécutée.
idL | idCli | idVehi | dateDeb | dateFin | 
9268 | 1345 | 245 | 2025-11-01 08:00:00 | None | 
(1 ligne(s))


#### ✅ RÉSOLUTION

```sql
UPDATE Location SET dateFin = '2025-11-03 16:00:00' WHERE idL = 9268;
```

*Exécution dans le notebook :*

In [4]:
affiche("""
UPDATE Location SET dateFin = '2025-11-03 16:00:00' WHERE idL = 9268;
""")
affiche("SELECT * FROM Location WHERE idL = 9268")

Instruction exécutée.
idL | idCli | idVehi | dateDeb | dateFin | 
9268 | 1345 | 245 | 2025-11-01 08:00:00 | 2025-11-03 16:00:00 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [5]:
assert resultat("SELECT dateFin FROM Location WHERE idL = 9268") == [("2025-11-03 16:00:00",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Lors de l'insertion, la date de fin était inconnue : on a mis `null` (valeur spéciale « non renseigné »). Quand la voiture est rendue, on **met à jour** cet enregistrement : on ne crée pas une nouvelle location.
- Syntaxe : `UPDATE Relation SET attribut = nouvelle_valeur WHERE condition;`
- La condition `WHERE idL = 9268` utilise la **clé primaire** : on est sûr de ne modifier **que** cette location-là.
- ⚠️ **Erreur grave à éviter :** oublier le `WHERE`. `UPDATE Location SET dateFin = '2025-11-03 16:00:00';` modifierait **toutes** les locations de la base !
- ⚠️ `WHERE idCli = 1345` serait aussi une mauvaise idée : si Mme Dupont a déjà loué d'autres voitures, toutes ses locations seraient modifiées.
- La date s'écrit entre guillemets simples, au format `'AAAA-MM-JJ HH:MM:SS'` (16h → `16:00:00`).

---
# QUESTION 4

#### 📝 ÉNONCÉ

> 4. Compléter la requête SQL ci-dessous de manière à lister les immatriculations des véhicules de la société qui ont plus de 50 000 km.
> ```sql
> SELECT ??? FROM Véhicule WHERE ??? ;
> ```

#### ✅ RÉSOLUTION

```sql
SELECT immatr FROM Véhicule WHERE nbKm > 50000;
```

*Exécution dans le notebook :*

In [6]:
affiche("""
SELECT immatr FROM Véhicule WHERE nbKm > 50000;
""")

immatr | 
AA-123-AA | 
DD-012-DD | 
EE-345-EE | 
(3 ligne(s))


*Vérification (non demandée) :*

In [7]:
assert sorted(resultat("SELECT immatr FROM Véhicule WHERE nbKm > 50000")) == [("AA-123-AA",), ("DD-012-DD",), ("EE-345-EE",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Après `SELECT` : l'attribut à **afficher** (l'immatriculation `immatr`) ; après `WHERE` : la **condition** de filtre (le kilométrage `nbKm`).
- ⚠️ Un nombre s'écrit **sans espace ni point** : `50000`, et non `50 000` ou `50.000` (qui voudrait dire 50 !). Pas de guillemets non plus : `nbKm` est un entier.
- « Plus de 50 000 km » se traduit par `>` ; `>=` inclurait aussi les véhicules qui ont exactement 50 000 km. À l'écrit, les deux sont en général acceptés si c'est cohérent.

---
# QUESTION 5

#### 📝 ÉNONCÉ

> 5. Écrire une requête SQL qui permet de connaître le nombre de locations en cours.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Location WHERE dateFin IS NULL;
```

*Exécution dans le notebook :*

In [8]:
affiche("""
SELECT COUNT(*) FROM Location WHERE dateFin IS NULL;
""")

COUNT(*) | 
2 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [9]:
assert resultat("SELECT COUNT(*) FROM Location WHERE dateFin IS NULL") == [(2,)]

# Avec = NULL au lieu de IS NULL :
affiche("SELECT COUNT(*) FROM Location WHERE dateFin = NULL")

COUNT(*) | 
0 | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Une location est en cours quand le véhicule n'a pas encore été rendu**, c'est-à-dire quand sa date de fin n'est **pas encore renseignée** : `dateFin` vaut `NULL` (c'est exactement la situation de la question 3 avant la mise à jour).
- ⚠️ **Le piège : `IS NULL` et pas `= NULL`.** `NULL` signifie « valeur inconnue » ; le SGBD ne peut pas dire si une valeur inconnue est égale à quelque chose, donc `dateFin = NULL` n'est **jamais vrai** et la requête renvoie 0 (voir la vérification). Il faut écrire `IS NULL` (ou `IS NOT NULL` pour le contraire). C'est indiqué page 8 du polycopié.
- `COUNT(*)` donne le **nombre** de locations, comme demandé (et non leur liste).

---
# QUESTION 6

#### 📝 ÉNONCÉ

> 6. Écrire une requête SQL qui affiche le nombre de véhicules électriques que la société possède.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Véhicule
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique';
```

*Exécution dans le notebook :*

In [10]:
affiche("""
SELECT COUNT(*) FROM Véhicule
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique';
""")

COUNT(*) | 
3 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [11]:
assert resultat("""
SELECT COUNT(*) FROM Véhicule
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique';
""") == [(3,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Le type de moteur n'est **pas** dans la relation `Véhicule` : il est dans `Modèle`. Chaque véhicule indique son modèle par la clé étrangère `modele`, qui fait référence à `Modèle.idM`. On **joint** donc les deux relations.
- ⚠️ La condition de jointure relie la clé étrangère à la clé primaire qu'elle référence : `Véhicule.modele = Modèle.idM`, et non `Véhicule.idV = Modèle.idM`.
- La valeur exacte de `moteur` (`'électrique'`, avec l'accent) dépend des données. Dans une vraie base, on la vérifie d'abord avec `SELECT DISTINCT moteur FROM Modèle;`.
- Ici, 3 véhicules électriques : deux Renault (même modèle) et une Tesla. On compte bien des **véhicules**, et non des modèles.

---
# QUESTION 7

#### 📝 ÉNONCÉ

> 7. Quelle requête SQL permet d'extraire les adresses électroniques des clients qui ont réservé un véhicule électrique durant le mois de septembre 2025 ?

#### ✅ RÉSOLUTION

```sql
SELECT DISTINCT Client.email FROM Client
    JOIN Location ON Client.idC = Location.idCli
    JOIN Véhicule ON Location.idVehi = Véhicule.idV
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique'
      AND Location.dateDeb >= '2025-09-01 00:00:00'
      AND Location.dateDeb < '2025-10-01 00:00:00';
```

*Exécution dans le notebook :*

In [12]:
affiche("""
SELECT DISTINCT Client.email FROM Client
    JOIN Location ON Client.idC = Location.idCli
    JOIN Véhicule ON Location.idVehi = Véhicule.idV
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique'
      AND Location.dateDeb >= '2025-09-01 00:00:00'
      AND Location.dateDeb < '2025-10-01 00:00:00';
""")

email | 
paul.durand@exemple.fr | 
linh.nguyen@exemple.fr | 
(2 ligne(s))


*Vérification (non demandée) :*

In [13]:
assert sorted(resultat("""
SELECT DISTINCT Client.email FROM Client
    JOIN Location ON Client.idC = Location.idCli
    JOIN Véhicule ON Location.idVehi = Véhicule.idV
    JOIN Modèle ON Véhicule.modele = Modèle.idM
    WHERE Modèle.moteur = 'électrique'
      AND Location.dateDeb >= '2025-09-01 00:00:00'
      AND Location.dateDeb < '2025-10-01 00:00:00';
""")) == [("linh.nguyen@exemple.fr",), ("paul.durand@exemple.fr",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Suivre le chemin des clés étrangères :** l'adresse est dans `Client`, la date dans `Location`, le moteur dans `Modèle`. Pour passer de l'un à l'autre, on traverse les **quatre** relations : `Client` → `Location` → `Véhicule` → `Modèle`. Chaque `JOIN ... ON` relie une clé étrangère à la clé primaire qu'elle référence.
- **Le mois de septembre :** on compare des dates comme des nombres : `>= '2025-09-01 00:00:00'` (à partir du 1er septembre à minuit) **et** `< '2025-10-01 00:00:00'` (strictement avant le 1er octobre). Écrire `<= '2025-09-30'` serait une erreur : `'2025-09-30 15:00:00'` est **après** `'2025-09-30 00:00:00'`, donc les locations du 30 septembre dans la journée seraient oubliées.
- **`DISTINCT` :** Paul Durand a loué **deux** véhicules électriques en septembre ; sans `DISTINCT`, son adresse apparaîtrait deux fois.
- Hugo Moreau a loué en septembre, mais une voiture **essence** : il n'apparaît pas. Il a loué une électrique en **octobre** : il n'apparaît pas non plus.
- **Interprétation :** on a considéré les locations qui **commencent** en septembre. On pourrait aussi compter celles qui **se déroulent** en partie en septembre (commencées fin août, par exemple) ; il faudrait alors une condition sur `dateDeb` et sur `dateFin`.